# Voice and Picture Creation (OpenAI API)
Examples for generating images and audio using LLMs to create prompts/scripts, then converting via OpenAI APIs.

This demonstrates multimodal output generation: text-to-image and text-to-speech via LLM assistance.

## Setup and Imports

In [ ]:
# Install required packages (run in terminal if needed)
# pip install openai requests pillow

from dotenv import load_dotenv
load_dotenv(override=True)

from openai import OpenAI
import requests
from PIL import Image
from io import BytesIO

# helper is now copied into this same directory
from file_creation_utils import get_unique_path

openai = OpenAI()

## LLM-Generated Image Creation (Text-to-Image)

In [ ]:
def generate_image_prompt(theme):
    """Use LLM to generate a detailed image prompt."""
    messages = [
        {"role": "system", "content": "You are a creative prompt engineer for AI image generation."},
        {"role": "user", "content": f"Create a detailed, vivid prompt for an image about: {theme}. Include style, colors, composition, and mood."}
    ]
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()

def create_image_from_prompt(prompt, size="1024x1024"):
    """Generate image using DALL-E and save to a local output folder."""
    response = openai.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=size,
        quality="standard",
        n=1
    )
    image_url = response.data[0].url
    
    # Download and save image
    img_response = requests.get(image_url)
    img = Image.open(BytesIO(img_response.content))

    # use helper to pick unique filename
    file_path = get_unique_path("generated_image", ".png")
    img.save(file_path)
    return file_path

# Example usage
theme = "a futuristic city at sunset"
prompt = generate_image_prompt(theme)
print(f"Generated Prompt: {prompt}")

image_path = create_image_from_prompt(prompt)
print(f"Image saved to: {image_path}")
# to view the image you can:
Image.open(image_path).show()

## LLM-Generated Voice Creation (Text-to-Speech)

In [ ]:
def generate_voice_script(topic):
    """Use LLM to generate a short script for TTS."""
    messages = [
        {"role": "system", "content": "You are a scriptwriter for voice narration."},
        {"role": "user", "content": f"Write a 50-100 word engaging script about: {topic}. Make it conversational and natural for speech."}
    ]
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=150
    )
    return response.choices[0].message.content.strip()

def create_voice_from_script(script, voice="alloy", model="tts-1"):
    """Generate audio using OpenAI TTS and save to output folder."""
    response = openai.audio.speech.create(
        model=model,
        voice=voice,
        input=script
    )
    
    # save audio file using helper
    file_path = get_unique_path("generated_voice", ".mp3")
    with open(file_path, "wb") as f:
        f.write(response.content)
    
    return file_path

# Example usage
topic = "the benefits of renewable energy"
script = generate_voice_script(topic)
print(f"Generated Script: {script}")

audio_file = create_voice_from_script(script)
print(f"Audio saved to: {audio_file}")
# Play with: import playsound; playsound.playsound(audio_file)

## Combined Example: Story with Image and Voice

In [ ]:
def create_multimodal_story(story_theme):
    """Generate a complete story with image and voice and save outputs in output folder."""
    # Generate story text
    messages = [
        {"role": "system", "content": "You are a storyteller."},
        {"role": "user", "content": f"Write a short, engaging story (100-200 words) about: {story_theme}."}
    ]
    story_response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300
    )
    story = story_response.choices[0].message.content.strip()
    
    # Generate image prompt from story
    image_prompt = generate_image_prompt(f"Illustrate this story: {story[:100]}...")
    
    # Create outputs
    image_path = create_image_from_prompt(image_prompt)
    audio_path = create_voice_from_script(story)
    
    return story, image_path, audio_path

# Example
theme = "a magical forest adventure"
story, img_path, audio_path = create_multimodal_story(theme)
print(f"Story: {story}")
Image.open(img_path).show()
print(f"Audio: {audio_path}")